# 외부 pod 호출하기
- 성공! 
    - ssh 백그라운드 실행명령 이용함
    - 도커파일 -> 이미지  -> pod 생성시 해당 이미지 이용하여 pod 실행에 따라 자동으로 vllm 서빙을 하도록 설정하는 방향으로 감

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://j93jaxl2w0uyaq-7860.proxy.runpod.net/v1",
    api_key="EMPTY"
)

# 테스트
response = client.chat.completions.create(
    model="google/gemma-3-1b-it",
    messages=[
        {"role": "user", "content": "안녕하세요! 파이썬으로 피보나치 수열을 생성하는 코드를 작성해주세요."}
    ],
    temperature=0.7,
    max_tokens=500
)

print(response.choices[0].message.content)

## 파이썬으로 피보나치 수열 생성 코드

다음은 파이썬으로 피보나치 수열을 생성하는 몇 가지 방법입니다.

**1. 반복문 사용 (가장 기본적인 방법)**

```python
def fibonacci_iterative(n):
  """
  반복문을 사용하여 피보나치 수열을 생성합니다.

  Args:
    n: 생성할 피보나치 수열의 항의 개수.

  Returns:
    피보나치 수열의 리스트.
  """
  if n <= 0:
    return []
  elif n == 1:
    return [0]
  else:
    list_fib = [0, 1]
    while len(list_fib) < n:
      next_fib = list_fib[-1] + list_fib[-2]
      list_fib.append(next_fib)
    return list_fib

# 예시
num_terms = 10
fib_sequence = fibonacci_iterative(num_terms)
print(f"피보나치 수열 (최대 {num_terms} 항): {fib_sequence}")
```

**코드 설명:**

*   `fibonacci_iterative(n)` 함수는 `n` 개의 피보나치 수열 항을 생성합니다.
*   `n`이 0보다 작거나 같으면 빈 리스트를 반환합니다.
*   `n`이 1이면 `[0]`을 반환합니다.
*   그렇지 않으면 `[0, 1]` 리스트를 초기화하고, `while` 루프를 사용하여 `list_fib` 리스트의 길이를 `n`으로 줄여나가며 피보나치 수열의 항을 생성합니다.
*   마지막에 `list_fib`를 반환합니다.

**2. 재귀 함수 사용 (가장 간단하지만 효율적이지 않음)**

```python
def fibonacci_recursive(n):
  """
  재귀 함수를 사용하여 피보나치 수열을 생성합니다.

  Args:
    n: 생성할 피보나치 수열의 항의 개수.

  Returns:
    피보나치

# RunPod vLLM 서버 구축 가이드

RunPod에서 vLLM을 사용하여 Gemma 3 1B 모델을 API 서버로 배포하는 방법

---

## 📋 목차

1. [개요](#개요)
2. [사전 준비](#사전-준비)
3. [RunPod Pod 배포](#runpod-pod-배포)
4. [vLLM 서버 설정](#vllm-서버-설정)
5. [API 사용법](#api-사용법)
6. [관리 및 운영](#관리-및-운영)
7. [문제 해결](#문제-해결)

---

## 개요

### 구축 결과
- **모델**: Google Gemma 3 1B (Instruction-tuned)
- **프레임워크**: vLLM 0.13.0
- **GPU**: NVIDIA RTX 4090
- **API 형식**: OpenAI 호환
- **접근성**: 외부 HTTPS 엔드포인트 제공

### 장점
- ✅ 빠른 추론 속도 (vLLM 최적화)
- ✅ OpenAI API와 호환되는 인터페이스
- ✅ 24/7 서비스 가능
- ✅ 팀원들과 API 공유 가능
- ✅ 사용하지 않을 때 과금 중지 가능

---

## 사전 준비

### 1. RunPod 계정
- [RunPod](https://www.runpod.io) 가입
- 결제 수단 등록

### 2. HuggingFace 토큰
1. [HuggingFace](https://huggingface.co) 가입
2. [Token 페이지](https://huggingface.co/settings/tokens) 접속
3. "New token" 클릭
4. Type: "Read" 선택
5. 토큰 복사 및 저장

### 3. SSH 키 생성 (Windows)

**Git Bash 또는 PowerShell**:
```bash
ssh-keygen -t ed25519 -f ~/.ssh/id_ed25519
# Enter 3번 (비밀번호 없이)
```

**공개키 확인**:
```bash
cat ~/.ssh/id_ed25519.pub
```

### 4. RunPod SSH 키 등록

1. [RunPod Settings](https://www.runpod.io/console/user/settings)
2. 좌측 메뉴 "Public Keys" 클릭
3. "Add Public Key" 클릭
4. 공개키 내용 붙여넣기
5. Save

---

## RunPod Pod 배포

### 1. GPU 선택
- 추천: **RTX 4090** (가성비 우수)
- 대안: RTX A6000, A100

### 2. 템플릿 설정

**Container Image**:
```
runpod/pytorch:2.1.0-py3.10-cuda11.8.0-devel-ubuntu22.04
```

**설정값**:
- Container Disk: 100GB
- Volume Disk: 100GB (선택사항)
- Expose HTTP Ports: `8888, 7860`
- Expose TCP Ports: `22`

### 3. 배포

"Deploy" 버튼 클릭 후 Pod가 **Running** 상태가 될 때까지 대기 (1-2분)

---

## vLLM 서버 설정

### 1. SSH 접속

**Pod Connect 탭**에서 SSH 명령어 확인:

```bash
ssh [pod-id]@ssh.runpod.io -i ~/.ssh/id_ed25519
```

예시:
```bash
ssh j93jaxl2w0uyaq-64410ab5@ssh.runpod.io -i ~/.ssh/id_ed25519
```

### 2. vLLM 설치

```bash
pip install vllm --upgrade --break-system-packages
```

**설치 확인**:
```bash
pip show vllm
# Version: 0.13.0 이상
```

### 3. HuggingFace 로그인

```bash
huggingface-cli login
```

토큰 입력 후 Enter

**성공 메시지**:
```
Login successful.
```

### 4. vLLM 서버 백그라운드 실행

**중요**: `nohup`을 사용하여 SSH 연결이 끊겨도 서버가 계속 실행되도록 설정

```bash
nohup python -m vllm.entrypoints.openai.api_server \
  --model google/gemma-3-1b-it \
  --host 0.0.0.0 \
  --port 7860 \
  --dtype auto \
  --max-model-len 8192 \
  --trust-remote-code > vllm.log 2>&1 &
```

### 5. 서버 시작 확인

**로그 확인** (1-3분 소요):
```bash
tail -f vllm.log
```

**성공 메시지**:
```
INFO:     Started server process
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:7860
```

`Ctrl+C`로 로그 보기 종료 (서버는 계속 실행됨)

### 6. 프로세스 확인

```bash
ps aux | grep vllm
```

**정상이면**:
```
root  12345  ... python -m vllm.entrypoints.openai.api_server
```

### 7. 외부 URL 확인

**RunPod Connect 탭**:
- HTTP services → Port 7860
- URL: `https://[pod-id]-7860.proxy.runpod.net`

**브라우저 테스트**:
```
https://[pod-id]-7860.proxy.runpod.net/v1/models
```

**성공 응답**:
```json
{
  "object": "list",
  "data": [{
    "id": "google/gemma-3-1b-it",
    "max_model_len": 8192,
    ...
  }]
}
```

---

## API 사용법

### 엔드포인트 정보

```
Base URL: https://[your-pod-id]-7860.proxy.runpod.net/v1
Model: google/gemma-3-1b-it
Max Tokens: 8192
API Key: EMPTY (필요 없음)
```

### Python 예제

**설치**:
```bash
pip install openai
```

**기본 사용**:
```python
from openai import OpenAI

# API 클라이언트 초기화
client = OpenAI(
    base_url="https://[your-pod-id]-7860.proxy.runpod.net/v1",
    api_key="EMPTY"
)

# 채팅 완성 요청
response = client.chat.completions.create(
    model="google/gemma-3-1b-it",
    messages=[
        {"role": "user", "content": "안녕하세요! 파이썬으로 피보나치 수열을 생성하는 코드를 작성해주세요."}
    ],
    temperature=0.7,
    max_tokens=500
)

print(response.choices[0].message.content)
```

**Streaming 응답**:
```python
stream = client.chat.completions.create(
    model="google/gemma-3-1b-it",
    messages=[
        {"role": "user", "content": "머신러닝이 무엇인가요?"}
    ],
    stream=True,
    temperature=0.7
)

for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
```

### LangChain 통합

```python
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    base_url="https://[your-pod-id]-7860.proxy.runpod.net/v1",
    api_key="EMPTY",
    model="google/gemma-3-1b-it",
    temperature=0.7
)

response = llm.invoke("한국의 전통 음식에 대해 설명해주세요.")
print(response.content)
```

### 팀원과 공유

**API 엔드포인트만 공유**:
```
https://[your-pod-id]-7860.proxy.runpod.net/v1
```

팀원들은 위 URL을 `base_url`에 입력하여 동일하게 사용 가능

---

## 관리 및 운영

### 서버 상태 확인

**SSH 접속 후**:
```bash
# 프로세스 확인
ps aux | grep vllm

# 포트 확인
ss -tulpn | grep 7860

# 로그 확인
tail -f vllm.log
```

### 서버 중지

```bash
pkill -f vllm
```

### 서버 재시작

```bash
nohup python -m vllm.entrypoints.openai.api_server \
  --model google/gemma-3-1b-it \
  --host 0.0.0.0 \
  --port 7860 \
  --dtype auto \
  --max-model-len 8192 \
  --trust-remote-code > vllm.log 2>&1 &
```

### Pod 관리

**사용하지 않을 때**:
1. RunPod 대시보드
2. Pod 카드 → "Stop" 버튼
3. 과금 중지 ✅

**재사용 시**:
1. RunPod 대시보드
2. Pod 카드 → "Start" 버튼
3. SSH 접속 후 vLLM 서버 재실행 필요

**주의**: Pod를 재시작하면 Pod ID와 URL이 변경될 수 있습니다!

### 비용 관리

**예상 비용** (RTX 4090 기준):
- 시간당: ~$0.50
- 일일 (24시간): ~$12
- 월간 (24/7): ~$360

**절약 팁**:
- 사용하지 않을 때 Pod Stop
- 개발 시간만 Running 상태 유지
- 필요시 더 저렴한 GPU 선택 (RTX A4000 등)

---

## 문제 해결

### 1. Port 7860이 "Not ready" 상태

**증상**: RunPod Connect 탭에서 Port 7860이 Ready가 안 됨

**해결**:
```bash
# 1. vLLM 서버 실행 확인
ps aux | grep vllm

# 2. 로그 확인
tail -f vllm.log

# 3. 브라우저에서 직접 테스트
https://[pod-id]-7860.proxy.runpod.net/v1/models

# 4. 1-2분 기다린 후 RunPod 페이지 새로고침
```

### 2. 404 또는 502 에러

**원인**: vLLM 서버가 실행되지 않음

**해결**:
```bash
# 프로세스 확인
ps aux | grep vllm

# 없으면 재실행
nohup python -m vllm... &
```

### 3. CUDA 메모리 부족

**증상**: 
```
CUDA out of memory
```

**해결**:
- 더 큰 GPU 선택 (A100)
- 또는 더 작은 모델 사용 (`gemma-2-2b-it`)

### 4. vLLM 버전 문제

**증상**:
```
Model architectures ['Gemma3ForCausalLM'] are not supported
```

**해결**:
```bash
pip install vllm --upgrade --break-system-packages
```

### 5. SSH 접속 실패

**증상**: Permission denied (publickey)

**해결**:
1. SSH 키 재생성
2. RunPod Settings에서 공개키 등록 확인
3. Pod 재시작

### 6. Git Bash 닫으면 서버 종료

**원인**: `nohup` 없이 실행

**해결**: 반드시 `nohup ... &` 사용

---

## 참고 자료

### 공식 문서
- [vLLM Documentation](https://docs.vllm.ai/)
- [RunPod Documentation](https://docs.runpod.io/)
- [Gemma Models](https://huggingface.co/google/gemma-3-1b-it)

### 주요 명령어 치트시트

```bash
# SSH 접속
ssh [pod-id]@ssh.runpod.io -i ~/.ssh/id_ed25519

# vLLM 설치
pip install vllm --upgrade --break-system-packages

# HuggingFace 로그인
huggingface-cli login

# vLLM 서버 실행 (백그라운드)
nohup python -m vllm.entrypoints.openai.api_server \
  --model google/gemma-3-1b-it \
  --host 0.0.0.0 \
  --port 7860 \
  --dtype auto \
  --max-model-len 8192 \
  --trust-remote-code > vllm.log 2>&1 &

# 로그 확인
tail -f vllm.log

# 프로세스 확인
ps aux | grep vllm

# 서버 중지
pkill -f vllm
```

---

## 성공 체크리스트

```
✅ RunPod 계정 생성
✅ HuggingFace 토큰 발급
✅ SSH 키 생성 및 등록
✅ Pod 배포 (RTX 4090)
✅ vLLM 0.13.0 설치
✅ HuggingFace 로그인
✅ Gemma 3 1B 서버 실행
✅ 백그라운드 실행 (nohup)
✅ 외부 URL 확인
✅ 브라우저 테스트 (200 OK)
✅ Python API 호출 성공
✅ 팀원과 URL 공유
```

---

## 라이선스 및 주의사항

- Gemma 3 모델은 Google의 라이선스 정책을 따릅니다
- 상업적 사용 시 [라이선스](https://ai.google.dev/gemma/terms) 확인 필요
- RunPod 사용 약관 준수
- API 남용 방지를 위한 적절한 사용량 관리 권장

---

**작성일**: 2024-12-27  
**버전**: 1.0  
**환경**: RunPod + vLLM 0.13.0 + Gemma 3 1B + RTX 4090